# NADOR: monitoring and experiment collection

This notebook introduces `MonitoringManagement` and `ExperimentRun`.
You can install the library without cloning or forking its repository.

Download this notebook, [monitoring.yaml](monitoring.yaml) and [queries.json](queries.json)
into the same folder. Use a Python 3.12+ Jupyter kernel.

This example adds monitoring to an existing SLICES orchestrator deployment.
Run these cells in your deployment notebook: its `ran` manager must represent the
successfully deployed core/RAN, with current inventory and SSH access. Leave UEs
unattached until monitoring is deployed. The full network setup is demonstrated
in the [RFsim examples](../monitoring/rfsim). Those examples use four VMs: node1
for the core/control plane, node2 for the RAN, node3 for UEs, and node4 for
monitoring and the traffic server. The ML configuration selects node4.

## 1. Install and import

Run installation before creating your network managers.

In [ ]:
%pip install --upgrade "git+https://github.com/yassir63/nador.git@main" --extra-index-url https://gitlab.inria.fr/api/v4/projects/70914/packages/pypi/simple


Restart the kernel after installation. Run your existing network setup to obtain
`ran`, then continue below. Installation is needed once per Python environment.

In [ ]:
import asyncio
from datetime import UTC, datetime
from pathlib import Path

from orchestrator.deployment import UEManagement
from nador import MonitoringDeployment, MonitoringManagement
from nador.artifacts import ExperimentRun

## 2. Configure monitoring

The accompanying profile monitors one AMF, SMF, UPF and gNB. Edit its component
names, containers and packet interfaces to match your deployment. Add a target
for each additional component you want to monitor. Host connections and namespaces
are resolved from `ran`; the YAML needs no SSH credentials.

This example enables latency and throughput probes. Grafana uses `admin` / `admin`.

In [ ]:
config = MonitoringDeployment.from_yaml_file("monitoring.yaml")
monitoring = MonitoringManagement(preceding=ran, config=config)

## 3. Deploy monitoring, then attach UEs

`deploy()` installs the monitoring services and injects the AMF/SMF sniffers.
It completes before UE attachment so the sniffers can observe session establishment.

In [ ]:
await monitoring.deploy()

Use your deployment's UE stage. For containerized UEs managed by the orchestrator,
the usual sequence is below. Physical UEs should connect using their normal procedure.

In [ ]:
ue = UEManagement(preceding=ran)
await ue.prepare()
await ue.install()

## 4. Run traffic and collect artifacts

Choose one UE component, its tunnel interface and an address reachable through
its 5G connection. The selected UE must contain `ping`.

The collector records the experiment interval and the named `ping` section.
`save_json()` saves the command output. On leaving the context, the collector
opens Prometheus access automatically, exports the queries in `queries.json`,
and closes the connection. Prometheus scrapes continuously during traffic.

This ping example demonstrates traffic and artifact collection; it does not
establish that TCP-based latency indicators have observations.

In [ ]:
UE_COMPONENT = "sim_ue"
UE_INTERFACE = "oaitun_ue1"
TARGET_IP = "<address reachable through the UE tunnel>"

run_id = datetime.now(UTC).strftime("%Y%m%dT%H%M%S%fZ")
result_dir = Path("results") / f"{run_id}-ping"

async with ExperimentRun(
    result_dir,
    name="ue-ping",
    monitoring=monitoring,
    queries_file="queries.json",
) as run:
    with run.section("ping"):
        report = await ue.run_command(
            command=["ping", "-I", UE_INTERFACE, "-c", "10", TARGET_IP],
            components=[UE_COMPONENT],
            quiet=True,
            flatten=False,
        )

    run.save_json("ping.json", report)
    await asyncio.sleep(2)  # Allow a final scrape after traffic ends.

print(f"Artifacts: {result_dir.resolve()}")

## 5. Inspect the result

Each run gets its own directory containing:

- `ping.json`: the UE command output.
- `experiment_metadata.json`: run status and any collection errors.
- `timeline_events.jsonl` and `timeline_summary.json`: recorded experiment sections.
- `prometheus_timeseries.csv` and `prometheus_export_summary.json`: exported metrics.
- `by_window/`: metric exports split by experiment and section.

Empty query results mean that the selected metric had no samples in the experiment
interval; consult the export summary. A completed traffic command alone does not
confirm that all monitoring metrics were available.

In [ ]:
print((result_dir / "experiment_metadata.json").read_text())

## Optional: remove monitoring

Run the following cell when finished. It removes monitoring without destroying
the core, RAN, UEs or locally saved experiment artifacts.

In [ ]:
# await monitoring.remove()